# 05 - Baseline Sentiment Models (Objective 6, part 1)
TF-IDF + classic ML models. These set the benchmark that BERT must beat.
Rule: **choose models on the validation set, touch the test set only once at the end.**

In [ ]:
%pip install -q scikit-learn pandas seaborn matplotlib pyarrow

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

Path("data/outputs").mkdir(parents=True, exist_ok=True)
train = pd.read_parquet("data/processed/train.parquet")
val = pd.read_parquet("data/processed/val.parquet")
test = pd.read_parquet("data/processed/test.parquet")
LABELS = ["negative", "neutral", "positive"]

## 1. Train every baseline and compare on the validation set

In [ ]:
tfidf = lambda: TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=50_000)

models = {
    "LogisticRegression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "NaiveBayes": MultinomialNB(),
    "LinearSVM": LinearSVC(class_weight="balanced"),
    "RandomForest": RandomForestClassifier(n_estimators=200, class_weight="balanced", n_jobs=-1, random_state=42),
}

rows, fitted = [], {}
for name, clf in models.items():
    pipe = make_pipeline(tfidf(), clf)
    pipe.fit(train["text_heavy"], train["label"])
    pred = pipe.predict(val["text_heavy"])
    fitted[name] = pipe
    rows.append({"model": name,
                 "accuracy": accuracy_score(val["label"], pred),
                 "f1_macro": f1_score(val["label"], pred, average="macro"),
                 "f1_weighted": f1_score(val["label"], pred, average="weighted")})

val_results = pd.DataFrame(rows).sort_values("f1_macro", ascending=False).round(3)
val_results

## 2. Final evaluation of the best baseline on the test set

In [ ]:
best_name = val_results.iloc[0]["model"]
best = fitted[best_name]
test_pred = best.predict(test["text_heavy"])

print("Best baseline:", best_name)
print(classification_report(test["label"], test_pred, labels=LABELS, digits=3))

cm = confusion_matrix(test["label"], test_pred, labels=LABELS)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=LABELS, yticklabels=LABELS)
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title(f"{best_name} - test confusion matrix")
plt.show()

baseline_test = {"model": best_name,
                 "accuracy": accuracy_score(test["label"], test_pred),
                 "f1_macro": f1_score(test["label"], test_pred, average="macro")}
pd.DataFrame([baseline_test]).to_csv("data/outputs/baseline_test_metrics.csv", index=False)